# Solution F1 · From a tree to Extra Trees: how much each ingredient gains

**Type**: Compare

## Objective

Compare four models **built one on top of the other** on three datasets (wines, tumors, digits): a single tree, bagging of trees (variance reduced by the average), the random forest (bagging + random features) and Extra Trees (random thresholds on top of random features). Measure accuracy in repeated cross-validation and time,
and quantify the gain of each ingredient.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_breast_cancer, load_digits, load_wine
from sklearn.ensemble import BaggingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier

datasets = {"wine": load_wine(return_X_y=True), "tumors": load_breast_cancer(return_X_y=True), "digits": load_digits(return_X_y=True)}
models = {
    "tree": lambda: DecisionTreeClassifier(random_state=0),
    "bagging": lambda: BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=0, n_jobs=-1),
    "forest": lambda: RandomForestClassifier(100, random_state=0, n_jobs=-1),
    "extra": lambda: ExtraTreesClassifier(100, random_state=0, n_jobs=-1),
}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=0)

## Commented solution

In [ ]:
rows = {}
for name, (X, y) in datasets.items():
    for model_name, create in models.items():
        r = cross_validate(create(), X, y, cv=cv)
        rows[(name, model_name)] = {"acc": r["test_score"].mean(), "t": r["fit_time"].mean()}
table = pd.DataFrame(rows).T
table["acc"].unstack()[["tree", "bagging", "forest", "extra"]].round(3)

In [ ]:
u = table["acc"].unstack().loc[list(datasets)]                       # rows in the order of the datasets
gains = pd.DataFrame({"averaging": u["bagging"] - u["tree"], "random_features": u["forest"] - u["bagging"], "random_thresholds": u["extra"] - u["forest"]})
order_holds = bool(((u["tree"] < u["bagging"]) & (u["bagging"] <= u["forest"] + 0.005) & (u["extra"] >= u["forest"] - 0.005)).all())
dominant_gain = gains.sum().idxmax()
print(gains.round(3))
print("order holds:", order_holds, "| dominant gain:", dominant_gain)

## Check

In [ ]:
with verify():
    check(table.shape == (12, 2) and list(table.columns) == ["acc", "t"], "12 rows (3 datasets x 4 models) and columns acc, t", "one row per (dataset, model) pair")
    check(list(gains.columns) == ["averaging", "random_features", "random_thresholds"] and list(gains.index) == list(datasets), "gains per dataset and per ingredient", "columns: averaging, random_features, random_thresholds")
    check((gains["averaging"] > 0.02).all() and dominant_gain == "averaging", "averaging the trees always gains over 2 points and it is the dominant gain", "bagging - tree")
    check(gains["random_features"].min() > -0.002 and gains["random_features"].max() < 0.03 and (gains["random_features"] >= 0).sum() >= 2, "random features gain little (0.5-2.5 points) and do not make things worse", "forest - bagging")
    check(order_holds is True and table.loc[("digits", "extra"), "acc"] > table.loc[("digits", "tree"), "acc"] + 0.1, "the order tree < bagging <= forest <= extra holds on all the datasets; on the digits the single tree is 10 points below Extra Trees", "compare the rows")
    check(table.loc[("wine", "tree"), "t"] < table.loc[("wine", "bagging"), "t"] / 3, "the single tree is much faster than the bagging (over three times)", "the cost grows with the number of trees")
done()

## What you learned

- **Averaging is the huge gain:** from the single tree to the bagging you gain 3-9 points (wine +4, tumors +3, digits +9.5): each tree is very variable and the average removes almost all the noise. It is 70-90% of the total gain.
- **Random features add little but cost nothing:** forest − bagging is +1.7 (wine), +0.6 (tumors), +2.4 (digits): more decorrelated trees, a slightly better average. With data that have few informative features the gain is larger.
- **Extra Trees** (random thresholds on top of random features) often does a little better still (+0.5-0.8): more bias per tree, less variance for the average, and it is also faster to train (no search for the optimal threshold). On the tumors the difference with the forest is half a point: in the noise.
- **The time** is proportional to the number of trees: a hundred trees cost a hundred times one tree (on the bagging with a single thread even more). The gain of 3-10 points is often worth the cost, but know that you are paying it.

## Common pitfall

Reporting "the random forest is the best" from a table where the difference with Extra Trees or with bagging is half a point. With repeated validation and differences of 0.5-1 point we are in the noise: the solid story is the jump from the single tree.

## Interview question

> *Click on a question to show its answer.*

<details><summary>What does the random forest add to bagging, and Extra Trees to the random forest?</summary>

Bagging reduces the variance by averaging trees on different bootstraps; the random forest adds the random subsampling of the features at each split to decorrelate the trees; Extra Trees also picks the threshold at random (it does not use the bootstrap by default): it increases the bias of each tree and reduces the variance of the average further,
and it is faster. In practice the big jump is from a single tree to an ensemble; the differences between the variants are a point or less and have to be checked on your own problem.

</details>